In [2]:
#Step 1: Load data and define the two comparison groups
import pandas as pd
import numpy as np
from scipy import stats

# Reload and rebuild order-level data (same as Phase 4)
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')

df = orders.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
df = df.merge(order_items[['order_id', 'product_id']], on='order_id', how='left')
df = df.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['is_return_proxy'] = ((df['review_score'] <= 2) | (df['order_status'] == 'canceled')).astype(int)

df = df.drop_duplicates(subset='order_id').reset_index(drop=True)
df = df.dropna(subset=['delivery_delay_days'])

# Check the distribution of delivery buffer to pick a sensible split point
print(df['delivery_delay_days'].describe())

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64


In [3]:
#We'll use the median (-12 days) as the natural split point
#Step 2: Define high-buffer vs. low-buffer groups

median_delay = df['delivery_delay_days'].median()

df['buffer_group'] = np.where(df['delivery_delay_days'] <= median_delay, 'high_buffer', 'low_buffer')

# high_buffer = more negative (delivered further ahead of estimate) = MORE buffer
# low_buffer = less negative / later = LESS buffer

print(df['buffer_group'].value_counts())
print(df.groupby('buffer_group')['delivery_delay_days'].mean())

buffer_group
high_buffer    52589
low_buffer     43887
Name: count, dtype: int64
buffer_group
high_buffer   -18.132765
low_buffer     -4.380568
Name: delivery_delay_days, dtype: float64


In [4]:
#Step 3: Compare return rates between the two groups (the "simulated A/B" comparison)
ab_summary = df.groupby('buffer_group')['is_return_proxy'].agg(['mean', 'count'])
ab_summary.columns = ['return_rate', 'order_count']
print(ab_summary)

# Statistical significance check (chi-square, since both variables are categorical: group x returned/not)
contingency = pd.crosstab(df['buffer_group'], df['is_return_proxy'])
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"\nChi-square statistic: {chi2_stat:.2f}")
print(f"P-value: {p_value:.6f}")

              return_rate  order_count
buffer_group                          
high_buffer      0.089049        52589
low_buffer       0.172921        43887

Chi-square statistic: 1515.04
P-value: 0.000000


In [6]:
#Step 4 
# Estimate: if we "shifted" low_buffer orders to match high_buffer's return rate, how many returns would be prevented?
low_buffer_orders = ab_summary.loc['low_buffer', 'order_count']
low_buffer_actual_returns = low_buffer_orders * ab_summary.loc['low_buffer', 'return_rate']
low_buffer_simulated_returns = low_buffer_orders * ab_summary.loc['high_buffer', 'return_rate']

returns_prevented = low_buffer_actual_returns - low_buffer_simulated_returns
pct_reduction = (returns_prevented / low_buffer_actual_returns) * 100

print(f"Actual returns in low-buffer group: {low_buffer_actual_returns:.0f}")
print(f"Simulated returns if low-buffer group matched high-buffer's rate: {low_buffer_simulated_returns:.0f}")
print(f"Estimated returns prevented: {returns_prevented:.0f}")
print(f"Relative reduction: {pct_reduction:.1f}%")

# Convert to a weekly estimate (dataset spans ~85 relevant weeks)
weeks_in_data = 85
returns_prevented_per_week = returns_prevented / weeks_in_data
print(f"\nEstimated returns prevented per week (if applied platform-wide): {returns_prevented_per_week:.1f}")

Actual returns in low-buffer group: 7589
Simulated returns if low-buffer group matched high-buffer's rate: 3908
Estimated returns prevented: 3681
Relative reduction: 48.5%

Estimated returns prevented per week (if applied platform-wide): 43.3
